In [6]:
import os
import glob
import pandas as pd

# ---------------------------------------------------------
# Valid penalties & importances
# ---------------------------------------------------------

VALID_PENALTIES = {
    "Fast_Shap",
    "Shapley",
    "Jacob_L1",
    "Jacob_F"
}

VALID_IMPORTANCES = {
    "Shapley",
    "Jacobian"
}

# ---------------------------------------------------------
# Small helpers
# ---------------------------------------------------------

def infer_importance_from_penalty(penalty: str) -> str | None:
    if penalty in {"Fast_Shap", "Shapley"}:
        return "Shapley"
    elif penalty in {"Jacob_F", "Jacob_L1"}:
        return "Jacobian"
    elif penalty == "Layer_Weight":
        return "Layer_Weight"
    else:
        return None


def parse_filename(path: str, expected_dataset: str | None = None) -> tuple | None:
    name = os.path.splitext(os.path.basename(path))[0]
    parts = name.split("_")

    offset = 0
    if parts[0].lower() in {"ablation", "simulation"}:
        offset = 1

    if len(parts) - offset < 5:
        return None

    dataset = parts[offset]
    series_str = parts[offset + 1]
    subject_str = parts[offset + 2]
    model = parts[offset + 3]
    penalty = "_".join(parts[offset + 4:])

    if expected_dataset is not None and dataset != expected_dataset:
        return None

    if penalty not in VALID_PENALTIES:
        return None

    importance = infer_importance_from_penalty(penalty)
    if importance is None or importance not in VALID_IMPORTANCES:
        return None

    try:
        series = int(series_str)
        subject = int(subject_str)
    except ValueError:
        return None

    return dataset, series, subject, model, penalty, importance

def pick_metric_columns(df: pd.DataFrame, importance: str) -> tuple[str, str] | None:
    candidates = [
        (f"AUROC_diag_{importance}", f"AUPRC_diag_{importance}"),
        (f"AUROC_{importance}", f"AUPRC_{importance}"),
        ("AUROC", "AUPRC"),
    ]
    for auroc_col, auprc_col in candidates:
        if auroc_col in df.columns and auprc_col in df.columns:
            return auroc_col, auprc_col
    return None

def detect_param_and_metric_cols(df: pd.DataFrame):
    """
    Everything except AUROC/AUPRC*/val_loss is treated as hyperparameter.
    """
    metric_prefixes = ("AUROC", "AUPRC")
    metric_cols = []
    param_cols = []
    for col in df.columns:
        if col == "val_loss":
            metric_cols.append(col)
        elif col.startswith(metric_prefixes):
            metric_cols.append(col)
        else:
            param_cols.append(col)
    return param_cols, metric_cols

def drop_error_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop rows containing 'Error(' or 'BrokenPipeError' anywhere.
    """
    if df.empty:
        return df

    # Convert everything to string so we can search easily
    s = df.astype(str)

    error_mask = s.apply(
        lambda row: row.str.contains("Error(", regex=False).any()
                    or row.str.contains("BrokenPipeError", regex=False).any(),
        axis=1,
    )

    n_bad = error_mask.sum()
    if n_bad > 0:
        print(f"[INFO] Dropping {n_bad} rows containing error messages")

    return df.loc[~error_mask].reset_index(drop=True)


# ---------------------------------------------------------
# Main logic: collect metrics & summarize
# ---------------------------------------------------------

def collect_file_metrics(root_dir: str, dataset: str, series: int) -> pd.DataFrame:
    pattern = os.path.join(root_dir, f"*{dataset}_{series}_*.csv")
    files = glob.glob(pattern)

    if not files:
        print(f"[INFO] No files found matching pattern: {pattern}")

    records = []

    for path in files:
        parsed = parse_filename(path, expected_dataset=dataset)
        if parsed is None:
            print(f"[INFO] Skipping file with unrecognized name: {os.path.basename(path)}")
            continue

        dataset_i, series_i, subject_i, model, penalty, importance = parsed
        if series_i != series:
            print(f"[INFO] Skipping {os.path.basename(path)} due to series mismatch.")
            continue

        # 1) Load CSV, skipping malformed lines
        try:
            df = pd.read_csv(path, on_bad_lines="skip")
        except Exception as e:
            print(f"[WARN] Could not load {path}: {e}")
            continue

        if df.empty:
            print(f"[WARN] {path} is empty; skipping.")
            continue

        # 2) Drop rows that contain error messages like BrokenPipeError(...)
        df = drop_error_rows(df)
        if df.empty:
            print(f"[WARN] {path} has only error rows after filtering; skipping.")
            continue

        # 3) Find AUROC/AUPRC columns for this importance
        metric_cols = pick_metric_columns(df, importance)
        if metric_cols is None:
            print(f"[WARN] {path} has no recognizable AUROC/AUPRC columns; skipping.")
            continue
        auroc_col, auprc_col = metric_cols

        # 4) Ensure metric columns are numeric
        df[auroc_col] = pd.to_numeric(df[auroc_col], errors="coerce")
        df[auprc_col] = pd.to_numeric(df[auprc_col], errors="coerce")

        # Drop rows where AUROC is NaN (so idxmax works)
        df_valid = df.dropna(subset=[auroc_col])
        if df_valid.empty:
            print(f"[WARN] {path}: all {auroc_col} values are NaN after cleaning; skipping.")
            continue

        # 5) Take the row with max AUROC
        best_idx = df_valid[auroc_col].idxmax()
        best_row = df_valid.loc[best_idx]

        # Split hyperparameters vs metrics (optional, but you had it)
        param_cols, _ = detect_param_and_metric_cols(df_valid)

        record = {
            "dataset": dataset_i,
            "series": series_i,
            "subject": subject_i,
            "model": model,
            "penalty": penalty,
            "importance": importance,
            "max_AUROC": best_row[auroc_col],
            "max_AUPRC": best_row[auprc_col],
        }

        # Add hyperparameters from the best row
        for col in param_cols:
            record[col] = best_row[col]

        records.append(record)

    if not records:
        return pd.DataFrame()

    df_out = pd.DataFrame(records)

    # Ensure metric columns are numeric
    df_out["max_AUROC"] = pd.to_numeric(df_out["max_AUROC"], errors="coerce")
    df_out["max_AUPRC"] = pd.to_numeric(df_out["max_AUPRC"], errors="coerce")

    return df_out

def summarize_over_subjects(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each (dataset, series, model, penalty, importance),
    compute mean/std of max_AUROC and max_AUPRC over subjects.
    """
    if df.empty:
        return df

    df = df.copy()
    df["max_AUROC"] = pd.to_numeric(df["max_AUROC"], errors="coerce")
    df["max_AUPRC"] = pd.to_numeric(df["max_AUPRC"], errors="coerce")
    df = df.dropna(subset=["max_AUROC", "max_AUPRC"])

    grouped = (
        df.groupby(["dataset", "series", "model", "penalty", "importance"])
        .agg(
            max_AUROC_mean=("max_AUROC", "mean"),
            max_AUROC_std=("max_AUROC", "std"),
            max_AUPRC_mean=("max_AUPRC", "mean"),
            max_AUPRC_std=("max_AUPRC", "std"),
            n_subjects=("subject", "nunique"),
        )
        .reset_index()
    )

    return grouped


# ---------------------------------------------------------
# Main entry point
# ---------------------------------------------------------

def main(subdir: str, dataset: str = "fMRI", series: int = 1):
    base_dir = "/Users/merlyn/Documents/Projects/Empirical Granger Causality/Code/SRNGC/server_results"

    root_dir = os.path.join(base_dir, subdir)
    out = os.path.join(base_dir, f"{subdir}_{dataset}_{series}.csv")

    print(f"[INFO] Root directory : {root_dir}")
    print(f"[INFO] Dataset/series: {dataset}/{series}")
    print(f"[INFO] Output file    : {out}")

    df = collect_file_metrics(root_dir, dataset, series)
    if df.empty:
        print("No matching files or no valid metrics found.")
        return

    summary = summarize_over_subjects(df)

    print("=== Summary over subjects (mean ± std of max metrics) ===")
    print(summary.to_string(index=False))

    # summary.to_csv(out, index=False)
    # print(f"\nSaved summary to {out}")
    
    return summary

In [7]:
combined_summary = pd.concat(
    [main('simulation_VAR3', 'VAR3', i) for i in range(1, 5)],
    ignore_index=True
)
# combined_summary.to_csv('./simulation_VAR3.csv')

[INFO] Root directory : /Users/merlyn/Documents/Projects/Empirical Granger Causality/Code/SRNGC/server_results/simulation_VAR3
[INFO] Dataset/series: VAR3/1
[INFO] Output file    : /Users/merlyn/Documents/Projects/Empirical Granger Causality/Code/SRNGC/server_results/simulation_VAR3_VAR3_1.csv
=== Summary over subjects (mean ± std of max metrics) ===
dataset  series       model   penalty importance  max_AUROC_mean  max_AUROC_std  max_AUPRC_mean  max_AUPRC_std  n_subjects
   VAR3       1 ResidualMLP Fast_Shap    Shapley        0.999810       0.000426        0.999583       0.000932           5
   VAR3       1 ResidualMLP   Jacob_F   Jacobian        0.998000       0.002702        0.995981       0.005055           5
   VAR3       1 ResidualMLP  Jacob_L1   Jacobian        0.999238       0.001452        0.998256       0.003332           5
   VAR3       1 ResidualMLP   Shapley    Shapley        0.998857       0.001801        0.997612       0.003632           5
[INFO] Root directory : /Users/m